# 13.2 · XGBoost on Top-N Features

**Purpose**: Train an XGBoost regression model restricted to the top N
features by LightGBM gain importance (from notebook 13).  Useful for
identifying the minimal feature set needed for competitive performance.

| I/O | Path |
|-----|------|
| Input  | `data/processed/train.parquet` |
| Input  | `data/processed/val.parquet` |
| Input  | `model/lgbm_atd_model.pkl` |
| Input  | `model/model_metadata.json` |
| Output | `model/xgb_top{N}_model.pkl` |
| Output | `model/xgb_top{N}_metadata.json` |
| Output | `data/processed/val_predictions_xgb.parquet` |

**Configure** `TOP_N` in the Setup cell to change feature budget.

---
## 1 · Setup

In [ ]:
import json
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from pathlib import Path
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120})

SEED              = 42
SLA_THRESHOLD_MIN = 45
TOP_N             = 15   # <-- change this to adjust feature budget

TRAIN_PATH    = Path('../data/processed/train.parquet')
VAL_PATH      = Path('../data/processed/val.parquet')
LGBM_MODEL    = Path('../model/lgbm_atd_model.pkl')
LGBM_META     = Path('../model/model_metadata.json')
MODEL_DIR     = Path('../model')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f'XGBoost version : {xgb.__version__}')
print(f'TOP_N           : {TOP_N}')

---
## 2 · Load Data & LightGBM Importance

In [ ]:
with open(LGBM_META, 'r', encoding='utf-8') as fh:
    lgbm_meta = json.load(fh)

TARGET       = lgbm_meta['target']
CAT_FEATURES = lgbm_meta['cat_features']
LGBM_COLS    = lgbm_meta['feature_cols']

df_train = pd.read_parquet(TRAIN_PATH)
df_val   = pd.read_parquet(VAL_PATH)

for col in CAT_FEATURES:
    for split in [df_train, df_val]:
        if col in split.columns:
            split[col] = split[col].astype('category')

y_train = df_train[TARGET].values
y_val   = df_val[TARGET].values

lgbm_model = joblib.load(LGBM_MODEL)

print(f'Train rows : {len(df_train):,}')
print(f'Val rows   : {len(df_val):,}')
print(f'LGBM cols  : {len(LGBM_COLS)}')

---
## 3 · Select Top-N Features by LightGBM Gain

In [ ]:
importance_df = pd.DataFrame({
    'feature': lgbm_model.feature_name(),
    'gain':    lgbm_model.feature_importance('gain'),
}).sort_values('gain', ascending=False).reset_index(drop=True)

top_features = importance_df.head(TOP_N)['feature'].tolist()

# Identify which top features are categorical
top_cat = [f for f in top_features if f in CAT_FEATURES]
top_num = [f for f in top_features if f not in CAT_FEATURES]

print(f'Top {TOP_N} features selected:')
print(f'  Numeric     : {len(top_num)}')
print(f'  Categorical : {len(top_cat)} {top_cat}')
print()
print(importance_df.head(TOP_N).to_string(index=False))

# Visualise
fig, ax = plt.subplots(figsize=(10, 5))
subset = importance_df.head(TOP_N).iloc[::-1]
colors = ['#06C167' if f in top_features else '#cccccc'
          for f in subset['feature']]
ax.barh(subset['feature'], subset['gain'],
        color=colors, edgecolor='none')
ax.set_title(
    f'Top {TOP_N} Features by LightGBM Gain'
    ' (selected for XGBoost)',
    fontweight='bold',
)
ax.set_xlabel('Total Gain')
plt.tight_layout()
plt.show()

---
## 4 · Prepare XGBoost Datasets

In [ ]:
X_train = df_train[top_features].copy()
X_val   = df_val[top_features].copy()

# Align category levels so val uses the same encoding as train
for col in top_cat:
    X_val[col] = pd.Categorical(
        X_val[col],
        categories=X_train[col].cat.categories,
    )

# XGBoost DMatrix with native categorical support
dtrain = xgb.DMatrix(
    X_train, label=y_train,
    enable_categorical=True,
)
dval = xgb.DMatrix(
    X_val, label=y_val,
    enable_categorical=True,
)

print(f'DMatrix train : {dtrain.num_row():,} x {dtrain.num_col()}')
print(f'DMatrix val   : {dval.num_row():,} x {dval.num_col()}')

---
## 5 · Train XGBoost

`reg:absoluteerror` mirrors LightGBM's `regression_l1` — both minimise
MAE — giving a fair apples-to-apples comparison.

In [ ]:
params = {
    'objective':        'reg:absoluteerror',
    'eval_metric':      ['mae', 'rmse'],
    'learning_rate':    0.05,
    'max_depth':        6,
    'min_child_weight': 50,
    'subsample':        0.8,
    'colsample_bytree': 0.8,
    'reg_alpha':        0.1,
    'reg_lambda':       0.1,
    'tree_method':      'hist',
    'seed':             SEED,
    'verbosity':        0,
}

evals_result = {}
print('Training XGBoost (reg:absoluteerror, early stopping on val MAE)...')

xgb_model = xgb.train(
    params,
    dtrain,
    num_boost_round=2000,
    evals=[(dtrain, 'train'), (dval, 'valid')],
    early_stopping_rounds=50,
    evals_result=evals_result,
    verbose_eval=100,
)

print(f'\nBest iteration : {xgb_model.best_iteration}')
print(f'Best val MAE   : {xgb_model.best_score:.4f}')

---
## 6 · Training Curve

In [ ]:
train_mae = evals_result['train']['mae']
val_mae   = evals_result['valid']['mae']
rounds    = range(1, len(train_mae) + 1)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(rounds, train_mae, label='Train MAE',
        color='steelblue', linewidth=1)
ax.plot(rounds, val_mae, label='Val MAE',
        color='#FF4B4B', linewidth=1)
ax.axvline(
    xgb_model.best_iteration,
    color='#06C167', linestyle='--', linewidth=1.2,
    label=f'Best iter {xgb_model.best_iteration}',
)
ax.set_xlabel('Boosting Round')
ax.set_ylabel('MAE (min)')
ax.set_title(
    f'XGBoost Training Curve — Top {TOP_N} Features',
    fontweight='bold',
)
ax.legend()
plt.tight_layout()
plt.show()

---
## 7 · Evaluate & Compare to LightGBM

In [ ]:
def compute_mape(
    y_true: np.ndarray,
    y_pred: np.ndarray,
) -> float:
    """MAPE excluding zero actuals."""
    mask = y_true > 0
    return float(
        np.mean(
            np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])
        ) * 100
    )


def evaluate_model(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    label: str = 'Model',
) -> dict:
    """Compute MAE, RMSE, MAPE, R², SLA-hit-rate."""
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mape = compute_mape(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    sla_actual    = float(np.mean(y_true <= SLA_THRESHOLD_MIN) * 100)
    sla_predicted = float(np.mean(y_pred <= SLA_THRESHOLD_MIN) * 100)
    return {
        'label':        label,
        'MAE':          round(mae, 3),
        'RMSE':         round(rmse, 3),
        'MAPE':         round(mape, 2),
        'R2':           round(r2, 4),
        'SLA_actual_%': round(sla_actual, 2),
        'SLA_pred_%':   round(sla_predicted, 2),
    }


pred_xgb = np.clip(
    xgb_model.predict(
        dval,
        iteration_range=(0, xgb_model.best_iteration),
    ),
    a_min=0, a_max=None,
)

result_xgb = evaluate_model(
    y_val, pred_xgb,
    f'XGBoost top-{TOP_N}',
)

# Pull LightGBM val metrics from saved metadata
result_lgbm = {
    'label': f'LightGBM (all {len(LGBM_COLS)} feats)',
    'MAE':   lgbm_meta['val_mae'],
    'RMSE':  lgbm_meta['val_rmse'],
    'MAPE':  lgbm_meta['val_mape'],
    'R2':    lgbm_meta['val_r2'],
    'SLA_actual_%': None,
    'SLA_pred_%':   None,
}

comparison = pd.DataFrame(
    [result_lgbm, result_xgb]
).set_index('label')

print(f'── Val Performance: XGBoost top-{TOP_N} vs LightGBM ──')
print(comparison.to_string())

mae_delta = result_xgb['MAE'] - lgbm_meta['val_mae']
print(
    f'\nXGBoost MAE delta vs LightGBM: '
    f'{mae_delta:+.3f} min '
    f'({mae_delta / lgbm_meta["val_mae"] * 100:+.1f}%)'
)
print(
    f'Feature budget: {TOP_N} / {len(LGBM_COLS)} '
    f'({TOP_N / len(LGBM_COLS) * 100:.0f}% of original set)'
)

---
## 8 · Predictions vs Actuals

In [ ]:
CLIP = 120  # cap display at 120 min for readability

actual  = np.clip(y_val,   a_min=0, a_max=CLIP)
predicted = np.clip(pred_xgb, a_min=0, a_max=CLIP)
residuals = y_val - pred_xgb

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── 1 · Hexbin: predicted vs actual ──────────────────────────────────────
hb = axes[0].hexbin(
    actual, predicted,
    gridsize=60, cmap='YlGn', mincnt=1,
)
axes[0].plot(
    [0, CLIP], [0, CLIP],
    color='#FF4B4B', linewidth=1.5, linestyle='--', label='Perfect',
)
fig.colorbar(hb, ax=axes[0], label='Trip count')
axes[0].set_xlabel('Actual ATD (min, clipped 120)')
axes[0].set_ylabel('Predicted ATD (min, clipped 120)')
axes[0].set_title('Predicted vs Actual', fontweight='bold')
axes[0].legend(fontsize=9)

# ── 2 · Residual distribution ─────────────────────────────────────────────
res_clip = np.clip(residuals, -60, 60)
axes[1].hist(
    res_clip, bins=80,
    color='#06C167', edgecolor='none', alpha=0.85,
)
axes[1].axvline(0, color='#FF4B4B', linestyle='--', linewidth=1.5)
axes[1].axvline(
    float(np.median(residuals)),
    color='#FFD700', linestyle='--', linewidth=1.2,
    label=f'Median {np.median(residuals):.1f} min',
)
axes[1].set_xlabel('Residual (actual − predicted, clipped ±60 min)')
axes[1].set_ylabel('Count')
axes[1].set_title('Residual Distribution', fontweight='bold')
axes[1].legend(fontsize=9)

# ── 3 · Residual vs actual (bias by ATD range) ───────────────────────────
axes[2].hexbin(
    actual, np.clip(residuals, -60, 60),
    gridsize=60, cmap='RdYlGn', mincnt=1,
)
axes[2].axhline(0, color='#FF4B4B', linestyle='--', linewidth=1.5)
axes[2].set_xlabel('Actual ATD (min, clipped 120)')
axes[2].set_ylabel('Residual (clipped ±60 min)')
axes[2].set_title('Residual vs Actual ATD\n(bias check by range)',
                  fontweight='bold')

plt.suptitle(
    f'XGBoost top-{TOP_N} — Val Set  '
    f'MAE={result_xgb["MAE"]:.2f} min  '
    f'R²={result_xgb["R2"]:.4f}',
    fontweight='bold', y=1.02,
)
plt.tight_layout()
plt.show()

# Summary stats
print(f'Residual mean   : {residuals.mean():+.3f} min')
print(f'Residual median : {np.median(residuals):+.3f} min')
print(f'Residual std    : {residuals.std():.3f} min')
print(
    f'Within ±5 min   : '
    f'{(np.abs(residuals) <= 5).mean()*100:.1f}%'
)
print(
    f'Within ±10 min  : '
    f'{(np.abs(residuals) <= 10).mean()*100:.1f}%'
)

---
## 8 · XGBoost Feature Importance

In [ ]:
xgb_imp = (
    pd.DataFrame(
        xgb_model.get_score(importance_type='gain').items(),
        columns=['feature', 'gain'],
    )
    .sort_values('gain', ascending=False)
    .reset_index(drop=True)
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(
    xgb_imp['feature'].iloc[::-1],
    xgb_imp['gain'].iloc[::-1],
    color='#06C167', edgecolor='none',
)
ax.set_title(
    f'XGBoost Feature Importance (Gain) — Top-{TOP_N} model',
    fontweight='bold',
)
ax.set_xlabel('Total Gain')
plt.tight_layout()
plt.show()

print(xgb_imp.to_string(index=False))

---
## 9 · Save Artifacts

In [ ]:
MODEL_PATH    = MODEL_DIR / f'xgb_top{TOP_N}_model.pkl'
META_PATH     = MODEL_DIR / f'xgb_top{TOP_N}_metadata.json'
VAL_PRED_PATH = Path('../data/processed/val_predictions_xgb.parquet')

# Model
joblib.dump(xgb_model, MODEL_PATH)
print(f'Model saved   → {MODEL_PATH}')

# Metadata
metadata = {
    'model':              'xgboost',
    'top_n':              TOP_N,
    'feature_cols':       top_features,
    'cat_features':       top_cat,
    'target':             TARGET,
    'importance_source':  'lgbm_gain',
    'best_iteration':     int(xgb_model.best_iteration),
    'val_mae':            result_xgb['MAE'],
    'val_rmse':           result_xgb['RMSE'],
    'val_mape':           result_xgb['MAPE'],
    'val_r2':             result_xgb['R2'],
    'lgbm_val_mae':       lgbm_meta['val_mae'],
    'mae_delta_vs_lgbm':  round(mae_delta, 4),
    'sla_threshold_min':  SLA_THRESHOLD_MIN,
    'split_date_val_start':  lgbm_meta.get('split_date_val_start'),
    'split_date_test_start': lgbm_meta.get('split_date_test_start'),
}
with open(META_PATH, 'w', encoding='utf-8') as fh:
    json.dump(metadata, fh, indent=2)
print(f'Metadata saved → {META_PATH}')

# Validation predictions
val_preds_df = pd.DataFrame({
    'workflow_uuid': df_val['workflow_uuid'].values,
    'ATD':           y_val,
    'ATD_predicted': pred_xgb,
})
val_preds_df.to_parquet(VAL_PRED_PATH, index=False, engine='pyarrow')
print(
    f'Val predictions → {VAL_PRED_PATH}'
    f'  ({len(val_preds_df):,} rows)'
)

print(
    f'\nSummary: XGBoost top-{TOP_N}  '
    f'MAE={result_xgb["MAE"]:.3f}  '
    f'RMSE={result_xgb["RMSE"]:.3f}  '
    f'R²={result_xgb["R2"]:.4f}'
)